In [ ]:
import json

dictionary_path = "../web crawler/IsTopic/files/new_topics_dictionary.txt"
with open(dictionary_path, mode="r") as f: 
    jsonn = json.loads(f.read())
    
dictionary = {}
search = [[] for i in range(len(jsonn))]
for i in range(len(jsonn)):
    dictionary[jsonn[i][0]] = i
#print(dictionary)

## Connect with mongodb

In [ ]:
import pymongo

myclient = pymongo.MongoClient("mongodb://localhost:27017/")
mydb = myclient["topic_collection"]
myindex = mydb["indexs"]
myorig = mydb["original"]
myckip = mydb["ckip"]

indexs = myindex.find()
othercount = 0
topics = []
for i in indexs:
    topic = myckip.find_one({"_id":i["ckip_id"]})
    topic["original_id"] = str(i["original_id"])
    topics.append(topic)

In [ ]:
def content_to_keys(topic):
    add=[]
    for info in topic["content"]:
        for word in info:
            if word['word'] in dictionary:
                if dictionary[word['word']] not in add:
                    add.append(dictionary[word['word']])
    return (topic["original_id"], add)

In [ ]:
from tqdm import tqdm
import multiprocessing as mp
pool = mp.Pool()
jobs = []
for topic in topics:    
    job = pool.apply_async(content_to_keys, (topic,))
    jobs.append(job)
for job in tqdm(jobs):    
    r = job.get()
    for index in r[1]:
        search[index].append(r[0])
pool.close()
pool.join()

In [ ]:
print(search[dictionary["森林"]])

In [ ]:
def flatten(l):
    return [item for sublist in l for item in sublist]

In [ ]:
from ckipnlp.pipeline import CkipPipeline, CkipDocument
pipeline = CkipPipeline(opts={'con_parser': {'username': "infoamplify", 'password': "2223pj[DB]"}})

In [ ]:
def Search(info):
    doc = CkipDocument(raw=info)#info is 文本
    result={}
    # Word Segmentation
    c = 0
    pipeline.get_ws(doc)
    for w in flatten(doc.ws.to_list()):
        if w in dictionary:
            item_ids = search[dictionary[w]]
            for item_id in item_ids:
                c += 1
                if item_id in result:
                    result[item_id] += 1
                else:
                    result[item_id] = 1
    result = list(sorted(result.items(), key=lambda item: item[1], reverse = True ))
    return result

In [ ]:
import tensorflow as tf
print(tf.__version__)

In [ ]:
from http.server import HTTPServer, BaseHTTPRequestHandler
import json
from urllib.parse import urlparse, unquote
from threading import Thread

class ResquestHandler(BaseHTTPRequestHandler):
    def do_GET(self):
        path = unquote(self.path)
        query = urlparse(path).query
        query_components = {}
        for qc in query.split("&"):
            d = qc.split("=")
            if len(d) == 2:
                query_components[d[0]] = d[1]
        #query_components = dict(qc.split("=") for qc in query.split("&"))
#         print('query_components:\n',query_components)
        if "search" in query_components:
            text = query_components["search"]
            result = Search(text)
#             print('result:\n',result)
            self.send_response(200)
            self.send_header('Content-type', 'application/json')
            self.end_headers()
            self.wfile.write(json.dumps(result).encode())
        else:
            self.send_response(403)    
server = HTTPServer(('', 3939), ResquestHandler)
server.serve_forever()